In [3]:
from IPython.display import Image

- references
    - https://fengyao.notion.site/off-policy-rl
        - https://docs.google.com/presentation/d/1TtHqvVY3C4M156vYLKSzooVWDB_rZR-BCpd7-3Qfoo0/edit?slide=id.g37644bc496b_0_42#slide=id.g37644bc496b_0_42
        - https://www.youtube.com/watch?v=b_wpvaHkylc&list=PL0etcga8W9k1Q6RoVQTRtQyxdofg5RU_D
    - https://huggingface.co/blog/NormalUhr/rlhf-pipeline
- REINFORCE 的梯度计算公式 $\mathbb E_{\tau\sim \pi_\theta}[R(\tau)\nabla \log\pi_theta]$ 要求期望 $\mathbb E$ 必须是在当前分布 $\pi_\theta$ 下计算的
- veRL: All experiments sample 1024 trajectories (64 prompts × 16 responses = 1024) at each training step and use a learning rate of 1e-6. The `ppo_mini_batch_size` is set to 1024 and 256 for on-policy and off-policy experiments, respectively.

### q-learning vs. sarsa

- references
    - https://www.youtube.com/watch?v=o_g9JUMw1Oc&list=PLJV_el3uVTsODxQFgzMzPLa16h6B8kWM_&index=4
    - https://dilithjay.com/blog/q-learning-and-sarsa
    - pg: on policy
        - https://hrl.boyuai.com/chapter/2/%E7%AD%96%E7%95%A5%E6%A2%AF%E5%BA%A6%E7%AE%97%E6%B3%95
    - dqn: off policy
        - https://hrl.boyuai.com/chapter/2/dqn%E7%AE%97%E6%B3%95
            - `self.q_net`
            - `self.target_q_net`
        - 关注下 loss function: Minimize TD Error
        - 关注数据的利用
- behavior policy vs. target policy
    - behavior policy: used to take actions in environment
    - target policy: used to optimize decision making
- off-policy RL: can decouple data collection and training
    - 智能体可以“通过观察别人（或者自己过去）的行为来学习”，即使那些行为并非当前最优的。

In [5]:
Image(url='./figs/q-learning-sarsa.jpeg', width=500)

> SARSA 和 Q-learning 都是 TD 学习的具体算法。它们唯一的区别在于 如何构建 TD 目标值，也就是如何估计下一步的价值。

$$
Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha(\underbrace{R_{t+1}+\gamma Q(S_{t+1},A_{t+1})}_{\text{TD target}}-Q(S_t,A_t))
$$

- 直观对比二者的流程：核心在于 $A_{t+1}$ 的选择
    - SARSA 的工作流程（一个时间步）
        - 第1步：在状态 S_t 做决策
            - 你当前处于状态 S_t。
            - 你根据你当前的策略（比如 ε-greedy 策略，它有一定概率会随机探索，有很大概率会选择当前认为最优的动作），选择了一个动作 A_t。
        - 第2步：执行动作并观察结果
            - 你执行动作 A_t。
            - 环境发生变化，你获得了一个即时奖励 R_{t+1}，并且进入了下一个状态 S_{t+1}。
        - 第3步：在新的状态 S_{t+1} 准备下一个决策（这是关键！）
            - 现在，你已经身处全新的状态 S_{t+1}。
            - 你需要决定下一步要干什么。于是，你再次使用你的策略（**和第1步中完全相同的策略**），在 S_{t+1} 这个状态下，选择出了你将要执行的下一个动作 A_{t+1}。
                - 选择 $A_t, A_{t+1}$ 是同一个策略
        - 第4步：回顾与更新
            - 好了，到此为止，你已经集齐了更新所需的所有要素，形成了一个完整的序列：
                - (状态 S_t, 动作 A_t, 奖励 R_{t+1}, 下一个状态 S_{t+1}, 下一个动作 A_{t+1})
            - 这就是 SARSA 这个名字的由来。
            - 现在，你用这个完整的序列，回头去更新你在第1步做的那个决策的价值，也就是 Q(S_t, A_t)。
            - 你用来更新的公式中的 Q(S_{t+1}, A_{t+1})，使用的就是你在第3步中实际选择出来、并准备在下一步执行的那个动作 A_{t+1}。
    - 至于 Q-learning
        - $Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha(R_{t+1}+\gamma\max_aQ(S_{t+1},a)-Q(S_t,A_t))$
        - 它在更新 Q(S_t, A_t) 时，它不关心 在 S_{t+1} 状态下，根据当前策略实际会选择哪个动作。相反，它会环顾 S_{t+1} 状态下的所有可能动作，然后挑选出那个能带来最大Q值的动作 $\max_a Q(S_t,a)$，，并用这个最乐观、最理想化的价值来构建 TD 目标。
        - 它的“内心独白”是：“我不管我下一步实际上会怎么走（可能因为探索我会走一步臭棋），我在评估过去时，永远都假设我下一步会做出最完美的选择。”
        - 这就是为什么 Q-learning 是 离策略（Off-policy） 的。它用于评估的策略（永远取 max）和它用于执行动作的策略（ε-greedy）可以是不同的。它学习的是一条通往最优目标的捷径，哪怕它自己当前还在蹒跚学步、到处乱撞。
        - $L_{RL}(\phi) = \mathbb{E}_{(s_i, a_i, r_i, s_i') \sim D} \left[ \left( y_i - Q_\phi(s_i, a_i) \right)^2 \right]
$

### rollout-training mismatch

In [2]:
Image(url='./figs/hybrid_engine.png', width=500)

- expected
$$
\theta \leftarrow \theta + \mu \cdot 
\underbrace{\mathbb{E}_{a \sim \pi(\theta)}}_{\text{rollout}}
\Big[ R(a) \cdot 
\underbrace{\nabla_{\theta} \log \pi(a, \theta)}_{\text{training}} \Big]
$$
- implementation

$$
\theta \leftarrow \theta + \mu \cdot
\underbrace{\mathbb{E}_{a \sim {\color{red}{\pi_{\mathrm{vllm}}}}(\theta)}}_{\text{\color{red}{rollout}}}
\Big[ R(a) \cdot
\underbrace{\nabla_{\theta}\,\log {\color{blue}{\pi_{\mathrm{fsdp}}}}(a,\theta)}_{\text{\color{blue}{training}}}
\Big]
$$

### importance ratio

In [3]:
Image(url='./figs/verl_mismatch.png', width=600)

- `verl/trainer/ppo/ray_trainer.py`：
    - `self.actor_rollout_wg`: `ActorRolloutRefWorker` (verl/workers/fsdp_workers.py)
        - actor: $\pi_\theta$, rollout: vllm rollout engine, ref: $\pi_{\text{ref}}$
    - `gen_batch_output = self.actor_rollout_wg.generate_sequences(gen_batch)`
        - https://github.com/volcengine/verl/blob/0d4541f397828843525b3f3a7eadff03d56ff24c/verl/trainer/ppo/ray_trainer.py#L972
    - `old_log_prob = self.actor_rollout_wg.compute_log_prob(batch)`: $\pi_{\text{old}}$
        - 此时：$\pi_\theta=\pi_\text{old}$，一个 batch 只算一次，在 batch 的 ppo training loop 内不会改变
        - https://github.com/volcengine/verl/blob/0d4541f397828843525b3f3a7eadff03d56ff24c/verl/trainer/ppo/ray_trainer.py#L1029
        - **recompute** `old_log_probs`: https://github.com/volcengine/verl/blob/0d4541f397828843525b3f3a7eadff03d56ff24c/verl/trainer/ppo/ray_trainer.py#L1027
    - `ref_log_prob = self.actor_rollout_wg.compute_ref_log_prob(batch)`: $\pi_{\text{ref}}$
        - https://github.com/volcengine/verl/blob/0d4541f397828843525b3f3a7eadff03d56ff24c/verl/trainer/ppo/ray_trainer.py#L1051
        - compute reference log_prob
    - compute advantage ...
    - training & updating actor ($\pi_\theta$)
        - https://github.com/volcengine/verl/blob/0d4541f397828843525b3f3a7eadff03d56ff24c/verl/trainer/ppo/ray_trainer.py#L1107
        - actor_output = self.actor_rollout_wg.update_actor(batch)